You can download and run this notebook locally, or you can run it for free in a cloud environment using Colab or Sagemaker Studio Lab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/TCIA_MedImageTools.ipynb)

[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github.com/kirbyju/TCIA_Notebooks/blob/main/TCIA_MedImageTools.ipynb)

# Background
This notebook is designed to showcase the core functionalities of Med-ImageTools using datasets from The Cancer Imaging Archive (TCIA). The notebook will guide you through the following steps:

1. Installing [Med-ImageTools](https://pypi.org/project/med-imagetools/) and [tcia_utils](https://pypi.org/project/tcia-utils/)
2. Downloading and processing a sample TCIA dataset using `AutoPipeline` for deep learning segmentation
   
   i. Understanding outputs from `AutoPipeline` for segmentation
   
   ii. Understanding full outputs from `AutoPipeline`
   
3. Processing a sample TCIA dataset using `AutoPipeline` with radiotherapy data

# 1 Setup:  Installing Med-ImageTolls and tcia_utils
Med-ImageTools and tcia_utils are available on PyPI and can be installed using pip.

In [1]:
import requests
import pandas as pd
import os
import imgtools
from tcia_utils import nbia

# 2 Downloading and processing a sample TCIA dataset using `AutoPipeline` for deep learning segmentation

We're going to start off by downloading some sample data and then telling AutoPipeline where to find and save data.

For this example, let's assume you've already spent some time [browsing the data on TCIA](https://www.cancerimagingarchive.net/access-data) and decided that you're interested in working with the [Pancreatic-CT-CBCT-SEG](https://doi.org/10.7937/TCIA.ESHQ-4D90) collection.  

**Note:** If you're interested in learning more about the functionality of **tcia_utils** for querying and downloading data please check out https://github.com/kirbyju/TCIA_Notebooks for a more thorough explanation.  

In [2]:
# download series metadata for the Pancreatic-CT-CBCT-SEG collection
series = nbia.getSeries("Pancreatic-CT-CBCT-SEG", format = "df")

2025-10-20 10:05:53,982:INFO:Success - Token saved to global api_call_headers variable and expires at 2025-10-20 12:05:53.982507
2025-10-20 10:05:53,984:INFO:Accessing public data anonymously. To access restricted data use nbia.getToken() with your credentials.
2025-10-20 10:05:53,985:INFO:Calling getSeries with parameters {'Collection': 'Pancreatic-CT-CBCT-SEG'}


In [3]:
# sort the 'series' dataframe by PatientID
series_sorted = series.sort_values('PatientID')

# get the first 3 unique PatientIDs
first_3_ids = series_sorted['PatientID'].unique()[:3]

# filter the dataframe to include only rows for the first 3 PatientIDs
sample_series = series_sorted[series_sorted['PatientID'].isin(first_3_ids)]

sample_series

,SeriesInstanceUID,StudyInstanceUID,Modality,SeriesDate,SeriesDescription,BodyPartExamined,SeriesNumber,Collection,PatientID,Manufacturer,...,ImageCount,TimeStamp,LicenseName,LicenseURI,CollectionURI,FileSize,DateReleased,StudyDesc,StudyDate,ThirdPartyAnalysis
285,1.3.6.1.4.1.14519.5.2.1.3094364563653665937958...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,CT,2012-07-19 00:00:00.0,NaN,ABDOMEN,5,Pancreatic-CT-CBCT-SEG,Pancreas-CT-CB_001,Varian Medical Systems,...,93,2022-08-11 13:02:12.0,Creative Commons Attribution 4.0 International...,https://creativecommons.org/licenses/by/4.0/,https://doi.org/10.7937/TCIA.ESHQ-4D90,48997226,2022-08-23 00:00:00.0,PANCREAS,2012-07-06 00:00:00.0,NO
241,1.3.6.1.4.1.14519.5.2.1.1873504185020096547520...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,CT,2012-07-28 00:00:00.0,NaN,ABDOMEN,8,Pancreatic-CT-CBCT-SEG,Pancreas-CT-CB_001,Varian Medical Systems,...,93,2022-08-11 13:02:19.0,Creative Commons Attribution 4.0 International...,https://creativecommons.org/licenses/by/4.0/,https://doi.org/10.7937/TCIA.ESHQ-4D90,49000938,2022-08-23 00:00:00.0,PANCREAS,2012-07-06 00:00:00.0,NO
108,1.3.6.1.4.1.14519.5.2.1.3023827908555824457224...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,CT,2012-07-06 00:00:00.0,"PANCREAS DI, iDose (3)",CHEST,201,Pancreatic-CT-CBCT-SEG,Pancreas-CT-CB_001,Philips,...,134,2021-09-16 12:49:02.0,Creative Commons Attribution 4.0 International...,https://creativecommons.org/licenses/by/4.0/,https://doi.org/10.7937/TCIA.ESHQ-4D90,70579828,2021-09-16 12:49:02.0,PANCREAS,2012-07-06 00:00:00.0,NO
216,1.3.6.1.4.1.14519.5.2.1.2982170419892875273227...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,RTSTRUCT,2015-11-12 00:00:00.0,BSCB_LL_LR_SDCB,ABDOMEN,1,Pancreatic-CT-CBCT-SEG,Pancreas-CT-CB_001,MIM Software Inc.,...,1,2021-09-16 12:42:59.0,Creative Commons Attribution 4.0 International...,https://creativecommons.org/licenses/by/4.0/,https://doi.org/10.7937/TCIA.ESHQ-4D90,2374672,2021-09-16 12:42:59.0,PANCREAS,2012-07-06 00:00:00.0,NO
83,1.3.6.1.4.1.14519.5.2.1.3090686406672815654659...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,CT,2012-07-19 00:00:00.0,Aligned resampled CB02,ABDOMEN,56094,Pancreatic-CT-CBCT-SEG,Pancreas-CT-CB_001,Varian Medical Systems / MIM Software,...,134,2021-09-16 12:48:25.0,Creative Commons Attribution 4.0 International...,https://creativecommons.org/licenses/by/4.0/,https://doi.org/10.7937/TCIA.ESHQ-4D90,70603434,2021-09-16 12:48:25.0,PANCREAS,2012-07-06 00:00:00.0,NO
116,1.3.6.1.4.1.14519.5.2.1.1813878355529485429185...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,RTSTRUCT,2015-11-09 00:00:00.0,BSPC_LL_LR_ROI_SDPC,ABDOMEN,1,Pancreatic-CT-CBCT-SEG,Pancreas-CT-CB_001,MIM Software Inc.,...,1,2021-09-16 12:18:00.0,Creative Commons Attribution 4.0 International...,https://creativecommons.org/licenses/by/4.0/,https://doi.org/10.7937/TCIA.ESHQ-4D90,3265634,2021-09-16 12:18:00.0,PANCREAS,2012-07-06 00:00:00.0,NO
26,1.3.6.1.4.1.14519.5.2.1.1027072244733702318999...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,CT,2012-07-28 00:00:00.0,Aligned CB07,ABDOMEN,46049,Pancreatic-CT-CBCT-SEG,Pancreas-CT-CB_001,Varian Medical Systems / MIM Software,...,134,2021-09-16 12:49:14.0,Creative Commons Attribution 4.0 International...,https://creativecommons.org/licenses/by/4.0/,https://doi.org/10.7937/TCIA.ESHQ-4D90,70602134,2021-09-16 12:49:14.0,PANCREAS,2012-07-06 00:00:00.0,NO
123,1.3.6.1.4.1.14519.5.2.1.2817663921020979728325...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,RTSTRUCT,2015-11-13 00:00:00.0,BSCB_LL_LR_SDCB,ABDOMEN,1,Pancreatic-CT-CBCT-SEG,Pancreas-CT-CB_001,MIM Software Inc.,...,1,2021-09-16 12:19:18.0,Creative Commons Attribution 4.0 International...,https://creativecommons.org/licenses/by/4.0/,https://doi.org/10.7937/TCIA.ESHQ-4D90,2384336,2021-09-16 12:19:18.0,PANCREAS,2012-07-06 00:00:00.0,NO
339,1.3.6.1.4.1.14519.5.2.1.2452239859660614261024...,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,RTDOSE,2012-07-13 00:00:00.0,Eclipse 

In [4]:
# Download the data
nbia.downloadSeries(sample_series, path="./tciaDownload/data", input_type = "df")

2025-10-20 10:06:03,158:INFO:Directory './tciaDownload/data' already exists.
2025-10-20 10:06:03,159:WARNING:Series 1.3.6.1.4.1.14519.5.2.1.309436456365366593795854640412150952125 already downloaded and unzipped.
2025-10-20 10:06:03,160:WARNING:Series 1.3.6.1.4.1.14519.5.2.1.187350418502009654752055781793285464352 already downloaded and unzipped.
2025-10-20 10:06:03,160:WARNING:Series 1.3.6.1.4.1.14519.5.2.1.302382790855582445722435410442490497846 already downloaded and unzipped.
2025-10-20 10:06:03,160:WARNING:Series 1.3.6.1.4.1.14519.5.2.1.298217041989287527322718266582029491750 already downloaded and unzipped.
2025-10-20 10:06:03,161:WARNING:Series 1.3.6.1.4.1.14519.5.2.1.309068640667281565465902342765215339781 already downloaded and unzipped.
2025-10-20 10:06:03,161:WARNING:Series 1.3.6.1.4.1.14519.5.2.1.18138783555294854291858006605381019315 already downloaded and unzipped.
2025-10-20 10:06:03,161:WARNING:Series 1.3.6.1.4.1.14519.5.2.1.102707224473370231899962923249595911866 alrea

In [5]:
!imgtools --help

Usage: imgtools [OPTIONS] COMMAND [ARGS]...

  A collection of tools for working with medical imaging data.

Options:
  -q, --quiet    Suppress all logging except errors, overrides verbosity
                 options.
  -v, --verbose  Increase verbosity of logging, overrides environment
                 variable. (0-3: ERROR, WARNING, INFO, DEBUG).
  --version      Show the version and exit.
  -h, --help     Show this message and exit.

AVAILABLE COMMANDS:

[CORE COMMANDS] Main subcommands for the med-imagetools package.
  index            Crawl DICOM directory and create a database index.
  interlacer       Visualize DICOM series relationships after indexing.
  autopipeline     Core utility to process messy DICOM data into...
  nnunet-pipeline  Process medical images in nnUNet format.

[UTILITIES] Tools for working with DICOM files.
  dicomfind  A tool to find DICOM files.
  dicomsort  Sorts DICOM files into directories based on their tags.
  dicomshow  Display and extract DICOM file m

### imgtools Index/Interlacer

In [6]:
INPUT_PATH  ="./tciaDownload"
OUTPUT_PATH ="./temp_output"

In [7]:
!imgtools -vvv index --dicom-dir $INPUT_PATH --force


10:06:07 [info     ] Starting DICOM crawl.          [imgtools] dicom_dir=PosixPath('tciaDownload') output_dir=PosixPath('.imgtools') dataset_name=None call=crawler.crawl:60
         [debug    ] Searching for DICOM files      [imgtools] directory=tciaDownload pattern=*.[dD][cC][mM] check_header=False case_sensitive=False search_input=None extension=[dD][cC][mM] call=dicom_find.filter_valid_dicoms:171
         [info     ] Found 1916 DICOM files in tciaDownload [imgtools] call=parse_dicoms.parse_dicom_dir:388
10:06:09 [info     ] Parsing all DICOMs took 1.2396 seconds [imgtools] call=timer_utils.__exit__:58
         [debug    ] Saved cache.                   [imgtools] crawl_cache=PosixPath('.imgtools/tciaDownload/crawl-cache.json') call=parse_dicoms.parse_dicom_dir:396
         [debug    ] Saved SOP map.                 [imgtools] sop_map_json=PosixPath('.imgtools/tciaDownload/sop_map.json') call=parse_dicoms.parse_dicom_dir:399
         [info     ] Mapping FrameOfReferenceUID to SeriesI

Running `AutoPipeline` creates a `.imgtools` folder in the dataset's parent directory.

```
parent_folder
└───.imgtools
│   ├── imgtools_dataset.csv
│   ├── imgtools_dataset.json
│   └── imgtools_dataset_edges.csv
│
└───dataset
    ├── patient-001
    ├── patient-002
    ...
```

There are three files in the `.imgtools` folder:

* `imgtools_dataset.csv` contains the metadata for the dataset
* `imgtools_dataset.json` contains the metadata for the dataset in JSON format
* `imgtools_dataset_edges.csv` contains the "edges" for the dataset.
    * An edge is a DICOM-DICOM pair that are connected based on the metadata.


In [8]:
from pathlib import Path
imgtools_files = list(Path("./.imgtools/tciaDownload").iterdir())

print(imgtools_files)

[PosixPath('.imgtools/tciaDownload/crawl-cache.json'), PosixPath('.imgtools/tciaDownload/sop_map.json'), PosixPath('.imgtools/tciaDownload/crawl_db.json'), PosixPath('.imgtools/tciaDownload/index.csv')]


This is what the crawled dataset looks like.
Each row represents a DICOM series (CT, MRI, PET, RTSTRUCT, SEG, RTDOSE, RTPLAN, etc).

In [9]:
df_crawl = pd.read_csv(Path("./.imgtools/tciaDownload/index.csv"), index_col=0)
df_crawl

,StudyInstanceUID,SeriesInstanceUID,SubSeries,Modality,ReferencedModality,ReferencedSeriesUID,instances,folder
PatientID,,,,,,,,
Pancreas-CT-CB_002,1.3.6.1.4.1.14519.5.2.1.1582046344163509315842...,1.3.6.1.4.1.14519.5.2.1.1058177309410879024400...,3,CT,NaN,NaN,228,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.1058...
Pancreas-CT-CB_002,1.3.6.1.4.1.14519.5.2.1.1582046344163509315842...,1.3.6.1.4.1.14519.5.2.1.6512363249234457542274...,1,CT,NaN,NaN,88,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.6512...
Pancreas-CT-CB_002,1.3.6.1.4.1.14519.5.2.1.1582046344163509315842...,1.3.6.1.4.1.14519.5.2.1.1768106710823996410134...,1,RTSTRUCT,CT,1.3.6.1.4.1.14519.5.2.1.1116851988950543841719...,1,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.1768...
Pancreas-CT-CB_003,1.3.6.1.4.1.14519.5.2.1.2027136189212788770525...,1.3.6.1.4.1.14519.5.2.1.9566682535928442256760...,1,CT,NaN,NaN,93,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.9566...
Pancreas-CT-CB_002,1.3.6.1.4.1.14519.5.2.1.1582046344163509315842...,1.3.6.1.4.1.14519.5.2.1.2114137508584631957882...,1,RTSTRUCT,CT,1.3.6.1.4.1.14519.5.2.1.1058177309410879024400...,1,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.2114...
Pancreas-CT-CB_003,1.3.6.1.4.1.14519.5.2.1.2027136189212788770525...,1.3.6.1.4.1.14519.5.2.1.2543830093577747739677...,1,RTDOSE,NaN,NaN,1,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.2543...
Pancreas-CT-CB_003,1.3.6.1.4.1.14519.5.2.1.2027136189212788770525...,1.3.6.1.4.1.14519.5.2.1.1004411337869179670896...,1,CT,NaN,NaN,93,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.1004...
Pancreas-CT-CB_003,1.3.6.1.4.1.14519.5.2.1.2027136189212788770525...,1.3.6.1.4.1.14519.5.2.1.1767644549555512228280...,1,CT,NaN,NaN,93,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.1767...
Pancreas-CT-CB_001,1.3.6.1.4.1.14519.5.2.1.2108734576221172452337...,1.3.6.1.4.1.14519.5.2.1.3094364563653665937958...,1,CT,NaN,NaN,93,tciaDownload/data/1.3.6.1.4.1.14519.5.2.1.3094...


In [11]:
!imgtools -vvv interlacer $INPUT_PATH 

10:06:28 [info     ] Initializing crawl.            [imgtools] call=interlacer.interlacer:86
         [info     ] Starting DICOM crawl.          [imgtools] dicom_dir=PosixPath('tciaDownload') output_dir=PosixPath('.imgtools') dataset_name=None call=crawler.crawl:60
         [info     ] .imgtools/tciaDownload/crawl-cache.json exists and force=False. Loading from file. [imgtools] call=parse_dicoms.parse_dicom_dir:376
         [info     ] .imgtools/tciaDownload/sop_map.json exists and force=False. Loading from file. [imgtools] call=parse_dicoms.parse_dicom_dir:379
         [info     ] Mapping FrameOfReferenceUID to SeriesInstanceUID took 0.0076 seconds [imgtools] call=timer_utils.__exit__:58
         [debug    ] Saved index CSV.               [imgtools] index_csv=PosixPath('.imgtools/tciaDownload/index.csv') call=parse_dicoms.parse_dicom_dir:443
         [debug    ] Saved crawl_db.                [imgtools] crawl_db_path=PosixPath('.imgtools/tciaDownload/crawl_db.json') call=parse_dicoms.

In [9]:
! imgtools -vvv nnunet-pipeline $INPUT_PATH $OUTPUT_PATH --modalities CT,RTSTRUCT --jobs 4 -ryaml ./example.yaml --mask-saving-strategy "sparse_mask"

16:07:43 [debug    ] ROI match map: {'LUNG_L': 'LUNG_L', 'LUNG_R': 'LUNG_R'} [imgtools] call=nnunet_pipeline.nnunet_pipeline:172
16:07:44 [debug    ] Using input directory name as dataset name: tciaDownload [imgtools] call=sample_input._set_default_dataset_name:140
         [debug    ] Pattern validation successful. [imgtools] keys=['DirType', 'SplitType', 'Dataset', 'SampleID'] formatted_pattern=%(DirType)s%(SplitType)s/%(Dataset)s_%(SampleID)s.nii.gz call=pattern_resolver.__init__:102
         [info     ] Pipeline initialized           [imgtools] call=nnunet_pipeline.__init__:145
         [info     ] Starting DICOM crawl.          [imgtools] dicom_dir=tciaDownload output_dir=.imgtools dataset_name=tciaDownload call=crawler.crawl:60
         [info     ] /home/joshua-siraj/Documents/BHKLAB/tcia_test/.imgtools/tciaDownload/crawl-cache.json exists and force=False. Loading from file. [imgtools] call=parse_dicoms.parse_dicom_dir:376
         [info     ] /home/joshua-siraj/Documents/BHKLAB/

In [10]:
!tree $OUTPUT_PATH


./temp_output
├── nnUNet_preprocessed
├── nnUNet_raw
│   └── Dataset001_tciaDownload
│       ├── Dataset001_tciaDownload_failed_20251017_160744.json
│       ├── Dataset001_tciaDownload_index.csv
│       ├── Dataset001_tciaDownload_index.csv.lock
│       ├── Dataset001_tciaDownload_successful_20251017_160744.json
│       ├── dataset.json
│       ├── imagesTr
│       │   ├── tciaDownload_0001_0000.nii.gz
│       │   ├── tciaDownload_0002_0000.nii.gz
│       │   ├── tciaDownload_0003_0000.nii.gz
│       │   ├── tciaDownload_0004_0000.nii.gz
│       │   ├── tciaDownload_0005_0000.nii.gz
│       │   ├── tciaDownload_0006_0000.nii.gz
│       │   ├── tciaDownload_0007_0000.nii.gz
│       │   ├── tciaDownload_0008_0000.nii.gz
│       │   └── tciaDownload_0009_0000.nii.gz
│       ├── labelsTr
│       │   ├── tciaDownload_0001.nii.gz
│       │   ├── tciaDownload_0002.nii.gz
│       │   ├── tciaDownload_0003.nii.gz
│       │   ├── tciaDownload_0004.nii.gz
│       │   ├── tciaDownload_0005.nii.gz


Let's take a look at the `dataset.json` file.

In [11]:
import json
json_path = (Path(OUTPUT_PATH) / "nnUNet_raw" / "Dataset001_tciaDownload" / "dataset.json")
json_data = json.load(open(json_path))
json_data

{'channel_names': {'0': 'CT'},
 'labels': {'LUNG_L': 1, 'LUNG_R': 2},
 'numTraining': 9,
 'file_ending': '.nii.gz',
 'licence': 'hands off!'}

In [13]:
import pandas as pd

df_path = (Path(OUTPUT_PATH) / "nnUNet_raw" / "Dataset001_tciaDownload" / "Dataset001_tciaDownload_index.csv")
df_data = pd.read_csv(df_path, index_col=0)
df_data


,AcquisitionNumber,AcquisitionTime,BodyPartExamined,CTDIvol,ContentDate,ContentTime,ContrastBolusAgent,ContrastBolusIngredient,ContrastBolusIngredientConcentration,ContrastBolusStartTime,...,origin,roi_matches.Background,roi_matches.LUNG_L,roi_matches.LUNG_R,saved_time,size,spacing,std,sum,variance
AcquisitionDate,,,,,,,,,,,,,,,,,,,,,
NaN,NaN,NaN,ABDOMEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"(-275.0, -178.10000610351, 85.9100036621093)",Background,LUNG_L,LUNG_R,2025-10-17:20:07:59,"(512, 512, 134)","(1.07421875, 1.07421875, 2.9999999999999973)",0.237642,1.196038e+06,0.056474
NaN,NaN,NaN,ABDOMEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"(-275.0, -178.10000610351, 85.9100036621093)",Background,LUNG_L,LUNG_R,2025-10-17:20:07:59,"(512, 512, 134)","(1.07421875, 1.07421875, 2.9999999999999973)",0.219430,9.828490e+05,0.048149
NaN,NaN,NaN,ABDOMEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"(-275.0, -178.1, 85.91)",Background,LUNG_L,LUNG_R,2025-10-17:20:08:00,"(512, 512, 134)","(1.07421875, 1.07421875, 3.0)",0.240654,1.245094e+06,0.057914
2012-07-19,1.0,19:11:08.853000,ABDOMEN,NaN,2015-11-12,15:34:54.334000,NaN,NaN,NaN,NaN,...,"(-275.0, -178.10000610351, 85.9100036621093)",Background,LUNG_L,LUNG_R,2025-10-17:20:08:01,"(512, 512, 134)","(1.07421875, 1.07421875, 2.9999999999999973)",231.691553,-3.309901e+10,53680.975925
2012-07-28,1.0,15:14:21.240000,ABDOMEN,NaN,2015-11-13,12:47:29.280000,NaN,NaN,NaN,NaN,...,"(-275.0, -178.10000610351, 85.9100036621093)",Background,LUNG_L,LUNG_R,2025-10-17:20:08:02,"(512, 512, 134)","(1.07421875, 1.07421875, 2.9999999999999973)",237.974831,-3.302746e+10,56632.020016
2012-07-08,1.0,11:24:19.333000,CHEST,NaN,2012-07-06,11:24:40.140000,NaN,NaN,NaN,NaN,...,"(-275.0, -178.1, 85.91)",Background,LUNG_L,LUNG_R,2025-10-17:20:08:01,"(512, 512, 134)","(1.07421875, 1.07421875, 3.0)",436.096146,-2.494947e+10,190179.848804
NaN,NaN,NaN,ABDOMEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"(-350.0, -302.0, 43.000061)",Background,LUNG_L,LUNG_R,2025-10-17:20:08:13,"(512, 512, 93)","(1.3671875, 1.3671875, 3.0)",0.248849,9.100950e+05,0.061926
2012-01-20,NaN,11:00:23.987000,ABDOMEN,NaN,2012-01-15,11:17:33.893000,NaN,NaN,NaN,NaN,...,"(-350.0, -302.0, 43.000061)",Background,LUNG_L,LUNG_R,2025-10-17:20:08:14,"(512, 512, 93)","(1.3671875, 1.3671875, 3.0)",378.925540,-1.916096e+10,143584.564951
NaN,NaN,NaN,ABDOMEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"(-350.0, -302.0, 43.0000610351562)",Background,LUNG_L,LUNG_R,2025-10-17:20:08:26,"(512, 512, 93)","(1.3671875, 1.3671875, 2.9999999999999982)",0.247100,8.985900e+05,0.061059


In [15]:
import os
from pathlib import Path

os.environ["nnUNet_raw"] = (Path(OUTPUT_PATH) / "nnUNet_raw").as_posix()
os.environ["nnUNet_preprocessed"] = (Path(OUTPUT_PATH) / "nnUNet_preprocessed").as_posix()
os.environ["nnUNet_results"] = (Path(OUTPUT_PATH) / "nnUNet_results").as_posix()

!nnUNetv2_extract_fingerprint -d 1 --verify_dataset_integrity

Dataset001_tciaDownload
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
100%|█████████████████████████████████████████████| 9/9 [00:08<00:00,  1.12it/s]
